# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 51), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.86 MiB | 9.66 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/flyrank-ml-internship


In [2]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB + HF token ready")

DuckDB + HF token ready


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# **My rule: A page gets a HIGH action score if it is OLD (365+ days) AND already**
gets decent visibility (impressions > 0) but its position is weak (position > 10).
Old + weak-position + some-visibility = highest refresh priority, because the
staleness signal (Signal 1) shows old content decays in CTR/position, and the
volume signal (Signal 2) shows visibility alone doesn't fix position — it needs
a refresh push.

Score = (age_weight × age_score) + (position_gap_weight × position_gap_score)
  - age_score: 0 (new <90d) / 0.5 (mid 90-365d) / 1 (old 365d+)
  - position_gap_score: normalized (position - 1) / max_position, capped at 1
    (higher position number = bigger gap = more opportunity)
  - weights: age_weight = 0.6, position_gap_weight = 0.4
    (staleness weighted higher — it's the stronger, more consistent signal)

Reason codes (whichever contributes more to the score, output as text):
  - "stale_content"      → age_score is the bigger contributor
  - "position_opportunity" → position_gap_score is the bigger contributor
  - "low_priority"       → both scores are low (new + already good position)

Action label:
  - score > 0.6  → "refresh_now"
  - score 0.3-0.6 → "monitor"
  - score < 0.3  → "no_action"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the ranked queue (writes the CSV)

score_query = """
SELECT
    c.content_hash_id,
    c.content_created_date,
    DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days,
    d.gsc_impressions,
    d.gsc_clicks,
    CASE WHEN d.gsc_impressions > 0 THEN d.gsc_sum_position::FLOAT / d.gsc_impressions ELSE NULL END as avg_position
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet') d
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
ON d.content_hash_id = c.content_hash_id
WHERE d.gsc_data_available IS TRUE
  AND d.gsc_impressions > 0
  AND c.content_created_date IS NOT NULL
"""

df = con.execute(score_query).df()

def age_score_fn(days):
    if days < 90:
        return 0.0
    elif days < 365:
        return 0.5
    else:
        return 1.0

df['age_score'] = df['content_age_days'].apply(age_score_fn)

max_position = df['avg_position'].max()
df['position_gap_score'] = ((df['avg_position'] - 1) / max_position).clip(0, 1)

age_weight = 0.6
position_gap_weight = 0.4

df['action_score'] = (age_weight * df['age_score']) + (position_gap_weight * df['position_gap_score'])

def reason_code_fn(row):
    age_contrib = age_weight * row['age_score']
    pos_contrib = position_gap_weight * row['position_gap_score']
    if age_contrib < 0.2 and pos_contrib < 0.2:
        return 'low_priority'
    elif age_contrib >= pos_contrib:
        return 'stale_content'
    else:
        return 'position_opportunity'

df['reason_code'] = df.apply(reason_code_fn, axis=1)

def action_label_fn(score):
    if score > 0.6:
        return 'refresh_now'
    elif score >= 0.3:
        return 'monitor'
    else:
        return 'no_action'

df['action_label'] = df['action_score'].apply(action_label_fn)

df_ranked = df.sort_values('action_score', ascending=False).reset_index(drop=True)
print(f"Total pages scored: {len(df_ranked)}")
print(df_ranked['action_label'].value_counts())
print(df_ranked.head(10)[['content_hash_id', 'content_age_days', 'avg_position', 'action_score', 'reason_code', 'action_label']])

import os
os.makedirs('work/outputs', exist_ok=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved to work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages scored: 3611061
action_label
monitor        2042037
no_action      1077130
refresh_now     491894
Name: count, dtype: int64
            content_hash_id  content_age_days  avg_position  action_score  \
0  content_aa376cef98a5fae8               375         447.0      0.958233   
1  content_39682b57561162fa               417         348.0      0.878715   
2  content_da708ca8845e8d4b               447         347.0      0.877912   
3  content_b4df40ce708e4536               417         339.0      0.871486   
4  content_a1174e866d52cb16               375         335.5      0.868675   
5  content_673012c87983e3ae               368         329.0      0.863454   
6  content_b0c4b0aea069fd1b               375         320.0      0.856225   
7  content_c00dff3870eba94d               375         300.0      0.840161   
8  content_b17838ccd31a43f1               390         273.0      0.818474   
9  content_18ca4cdf6f782d6f               371         269.0      0.815261   

     reason_code

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Top-10 review

top10 = df_ranked.head(10)[['content_hash_id', 'content_age_days', 'gsc_impressions', 'avg_position', 'action_score', 'reason_code', 'action_label']]
print(top10.to_string())

            content_hash_id  content_age_days  gsc_impressions  avg_position  action_score    reason_code action_label
0  content_aa376cef98a5fae8               375                1         447.0      0.958233  stale_content  refresh_now
1  content_39682b57561162fa               417                1         348.0      0.878715  stale_content  refresh_now
2  content_da708ca8845e8d4b               447                1         347.0      0.877912  stale_content  refresh_now
3  content_b4df40ce708e4536               417                1         339.0      0.871486  stale_content  refresh_now
4  content_a1174e866d52cb16               375                2         335.5      0.868675  stale_content  refresh_now
5  content_673012c87983e3ae               368                1         329.0      0.863454  stale_content  refresh_now
6  content_b0c4b0aea069fd1b               375                1         320.0      0.856225  stale_content  refresh_now
7  content_c00dff3870eba94d               375   

# **Top-10 Review:**

**Row 1 — content_aa376cef98a5fae8:**
- Action: refresh_now
- Why: Old (375d), position 447, reason_code = stale_content
- What would make it wrong: Only 1 impression — a single rare query landed
  at position 447. Not a reliable signal; this page may not need refresh at all.

**Row 2 — content_39682b57561162fa:**
- Action: refresh_now
- Why: Old (417d), position 348
- What would make it wrong: Only 1 impression — same low-confidence issue.

**Row 3 — content_da708ca8845e8d4b:**
- Action: refresh_now
- Why: Old (447d), position 347
- What would make it wrong: Only 1 impression — position is likely a fluke,
  not a stable ranking signal.

**Row 4 — content_b4df40ce708e4536:**
- Action: refresh_now
- Why: Old (417d), position 339
- What would make it wrong: Only 1 impression — unreliable.

**Row 5 — content_a1174e866d52cb16:**
- Action: refresh_now
- Why: Old (375d), position 335.5
- What would make it wrong: 2 impressions — slightly more data, but still too
  low to trust; position could shift heavily with a few more impressions.

**Row 6 — content_673012c87983e3ae:**
- Action: refresh_now
- Why: Old (368d), position 329
- What would make it wrong: Only 1 impression — noisy pick.

**Row 7 — content_b0c4b0aea069fd1b:**
- Action: refresh_now
- Why: Old (375d), position 320
- What would make it wrong: Only 1 impression — same low-confidence risk.

**Row 8 — content_c00dff3870eba94d:**
- Action: refresh_now
- Why: Old (375d), position 300
- What would make it wrong: Only 1 impression — unreliable.

**Row 9 — content_b17838ccd31a43f1:**
- Action: refresh_now
- Why: Old (390d), position 273
- What would make it wrong: Only 1 impression — unreliable.

**Row 10 — content_18ca4cdf6f782d6f:**
- Action: refresh_now
- Why: Old (371d), position 269
- What would make it wrong: Only 1 impression — unreliable.

Overall observation: All top-10 picks share the same weakness — extremely
low impressions (1-2), meaning the position values are statistical noise,
not stable rankings. The rule needs a minimum-impressions filter (e.g.
impressions >= 10) before trusting position_gap_score. This is a real
limitation of the current baseline rule and should be flagged in Section 4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak picks + leakage check

# Weak picks: refresh_now picks with very low impressions (unreliable position)
weak_picks = df_ranked[(df_ranked['action_label'] == 'refresh_now') & (df_ranked['gsc_impressions'] < 10)]
print(f"Weak picks (refresh_now with <10 impressions): {len(weak_picks)} out of {len(df_ranked[df_ranked['action_label']=='refresh_now'])} total refresh_now picks")
print(f"Percentage: {100 * len(weak_picks) / len(df_ranked[df_ranked['action_label']=='refresh_now']):.1f}%")

# Leakage check: confirm no product/health/priority flags or future-window data used
features_used = ['content_age_days', 'gsc_impressions', 'gsc_clicks', 'avg_position']
print(f"\nFeatures used in scoring: {features_used}")
print("Confirmed: no product flags (health_score, priority_score) used.")
print("Confirmed: data pulled only from month=2026-03 (no future window).")
print("Confirmed: label (is_declining_label) was NOT used as a feature — score is built independently from staleness + position gap only.")

Weak picks (refresh_now with <10 impressions): 202224 out of 491894 total refresh_now picks
Percentage: 41.1%

Features used in scoring: ['content_age_days', 'gsc_impressions', 'gsc_clicks', 'avg_position']
Confirmed: no product flags (health_score, priority_score) used.
Confirmed: data pulled only from month=2026-03 (no future window).
Confirmed: label (is_declining_label) was NOT used as a feature — score is built independently from staleness + position gap only.


# **Weak picks: **
202,224 out of 491,894 refresh_now picks (41.1%) have fewer than
10 impressions — meaning their avg_position is based on very little data and
is likely noisy/unreliable, not a stable ranking signal. This is a real
limitation of the current baseline rule. Fix for next iteration: add a
minimum-impressions filter (e.g. impressions >= 10) before trusting
position_gap_score, or down-weight position_gap_score when impressions are low.

Leakage check: Confirmed clean.
- No product-internal flags (health_score, priority_score) used as features.
- Data pulled only from month=2026-03 — no future window beyond the scoring date.
- The label used in ML-03 (is_declining_label, based on trend_direction) was
  NOT used anywhere in this score — this rule is built independently from
  raw staleness (content age) and position gap only.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decsion-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.